# 316 — Orchestrator & Direct API Usage

Demonstrates the multi-agent pipeline (Orchestrator → ComplianceAgent + InvestigationAgent)
via direct Python calls. The interactive chat UI has moved to  (Streamlit).

**Pre-loaded example questions** use real entity IDs from the data.

In [1]:
%run 311_agent_setup.ipynb

/Users/emilpastor/opt/miniconda3/envs/loanguard-ai/lib/python3.11/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Environment loaded.


2026-03-10 20:37:55,380 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


Connected to Neo4j.

Node counts in graph:
  {'labels': {'CommercialSecured': 2, 'Federal': 1, 'Address': 609, 'Residential': 582, 'Corporate': 31, 'Collateral': 466, 'ResidentialSecured': 464, 'Industry': 14, 'National': 5, 'PersonalTransaction': 610, 'Threshold': 133, 'CorporateCurrent': 6, 'Borrower': 628, 'BusinessTransaction': 175, 'SpecialAdminRegion': 1, 'BankAccount': 791, 'Chunk': 189, 'Section': 101, 'Jurisdiction': 7, 'Commercial': 27, 'Requirement': 219, 'Transaction': 173, 'LoanApplication': 466, 'Individual': 597, 'Regulation': 3, 'Officer': 19}}
  ✓  Borrower: 628
  ✓  LoanApplication: 466
  ✓  BankAccount: 791
  ✓  Transaction: 173
  ✓  Regulation: 3
  ✓  Section: 101
  ✓  Requirement: 219
  ✓  Threshold: 133
  ✓  Chunk: 189
Anthropic client ready. Model: claude-sonnet-4-6
OpenAI client ready.
Tool registry: 2 Neo4j MCP + 6 FastMCP = 8 total
  read-neo4j-cypher
  write-neo4j-cypher
  traverse_compliance_path
  retrieve_regulatory_chunks
  detect_graph_anomalies
  persis

In [2]:
from src.agent.orchestrator import Orchestrator

orchestrator = Orchestrator(tools=TOOLS, execute_tool_fn=execute_tool)
print("Orchestrator ready.")

2026-03-10 20:37:56,223 [INFO] src.agent.dispatcher: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-10 20:37:56,244 [INFO] src.agent.orchestrator: Graph regulations found: ['APG-223', 'APS-112', 'APS-220']


Orchestrator ready.


## Direct API usage

Run questions programmatically and inspect the structured .

In [3]:
resp = orchestrator.run('Find all transaction structuring patterns.')
print(f'Verdict:    {resp.verdict}')
print(f'Confidence: {resp.confidence}')
print(f'Findings:   {len(resp.findings)}')
print(f'Cypher:     {len(resp.cypher_used)} queries')
print(f'Answer (first 500 chars):{resp.answer[:500]}')

2026-03-10 20:37:57,526 [INFO] src.agent.orchestrator: [9fa491d6] Orchestrator: Find all transaction structuring patterns.
2026-03-10 20:38:02,827 [INFO] src.agent.orchestrator: [9fa491d6] Routing: {'intents': ['anomaly', 'investigation'], 'entity_ids': [], 'entity_types': ['Transaction'], 'regulations': [], 'run_anomaly_check': True, 'needs_compliance_agent': False, 'needs_investigation_agent': True}
2026-03-10 20:38:02,828 [INFO] src.agent.investigation_agent: InvestigationAgent: Find all transaction structuring patterns.
2026-03-10 20:38:10,036 [INFO] src.agent.investigation_agent: Tool: detect_graph_anomalies(['pattern_names'])
2026-03-10 20:38:10,037 [INFO] src.agent.dispatcher: Tool: detect_graph_anomalies | inputs: ['pattern_names']
2026-03-10 20:38:10,222 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-10 20:38:10,551 [INFO] src.graph.connection: Neo4j connection closed.
2026-03-10 20:38:10,553 [INFO] src.agent.investigation_agen

Verdict:    INFORMATIONAL
Confidence: 0.5
Findings:   5
Cypher:     2 queries
Answer (first 500 chars):A clear transaction structuring pattern has been identified centred on entity BRW-0602 (Hassan Martin). All 19 transactions are deliberately kept below the $10,000 threshold, which is a hallmark indicator of smurfing designed to evade AUSTRAC Threshold Transaction Reporting obligations. The pattern is compounded by Hassan Martin's pre-existing HIGH risk rating, unresolved source accounts across all transactions, and three active loan applications (LOAN-0452, LOAN-0453, LOAN-0454) that may be con


## Inspect findings

In [4]:
for f in resp.findings:
    print(f'[{f["severity"]}] {f["description"]}')

[HIGH] [HIGH]** All 19 transactions are individually sub-$10,000 — consistent with deliberate structuring to avoid AUSTRAC Threshold Transaction Reporting (TTR) obligations under the *Anti-Money Laundering and Counter-Terrorism Financing Act 2006*.
[HIGH] [HIGH]** Hassan Martin (BRW-0602) carries a pre-existing **HIGH risk rating** with a low credit score of 614, compounding the structuring concern.
[HIGH] [HIGH]** All three borrowers have active loan applications (LOAN-0452, LOAN-0453, LOAN-0454) — the structured funds may represent undisclosed deposit sources or loan repayment funds.
[MEDIUM] [MEDIUM]** Source accounts for all 19 transactions are **unresolved** (no FROM_ACCOUNT linked in the graph) — the origin of funds is unknown, which is a significant gap in the AML audit trail.
[MEDIUM] [MEDIUM]** The three clusters span different time windows (Aug, Oct, Nov 2025) and different banks, suggesting these may be coordinated or part of a broader network rather than isolated incidents.

## Inspect routing decision

In [5]:
import json
print(json.dumps(resp.routing, indent=2))

{
  "intents": [
    "anomaly",
    "investigation"
  ],
  "entity_ids": [],
  "entity_types": [
    "Transaction"
  ],
  "regulations": [],
  "run_anomaly_check": true,
  "needs_compliance_agent": false,
  "needs_investigation_agent": true
}
